# 📌 Model Context Protocol (MCP)

![Topic](https://img.shields.io/badge/Topic-MCP-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Agentic-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-August%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — MCP (Model Context Protocol) is an open standard, created by Anthropic, that lets an AI model connect to external tools and data (files, databases, apps like Slack or GitHub) through one common "language" instead of a custom integration for every pairing. It's an open standard introduced by Anthropic with the goal to standardize how AI applications connect with external tools, data sources, and systems.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install mcp` |


---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Before MCP, if you wanted an AI app to talk to Slack, GitHub, a database, and your local files, someone had to build four separate custom integrations — and every new AI app repeated that work. This is known as the "M×N problem": with M AI applications and N tools, you might need M×N different integrations, leading to duplicated effort and inconsistency.

The Model Context Protocol is an open standard for connecting AI assistants to the systems where data lives, including content repositories, business tools, and development environments. It was created at Anthropic and introduced in November 2024. Instead of every AI app needing its own connector for every tool, developers implement MCP once and unlock an entire ecosystem of integrations.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

MCP uses a client-server architecture: MCP clients live inside host applications like Claude or Cursor and make requests, while MCP servers expose the tools, resources, and prompts those applications can use.

1. You (or the LLM) trigger a need
2. The MCP client (built into the host app) formats that need as a request.
3. The request travels to an MCP server - a small program that wraps a specific tool or data source (GitHub, Slack, a local database, etc.). MCP supports two transports: stdio for local servers running as subprocesses, and HTTP with Server-Sent Events for remote or cloud-deployed servers, both using JSON-RPC 2.0 as the underlying communication protocol.
4. The server does the actual work - it queries GitHub's API, reads the file, runs the SQL query.
5. The result flows back through the server to the client to the LLM, which uses it to write its answer.

Anthropic even open-sourced ready-made servers: pre-built MCP servers for popular systems like Google Drive, Slack, GitHub, Git, Postgres, and Puppeteer.

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **One integration, many tools** | write once, and it works with any MCP-compatible app instead of custom code per pairing. |
| 🟢 | **Model/vendor independence** | MCP offers vendor flexibility, making it easier to switch between LLM providers without significant code changes. |
| 🟢 | **Built for complex, multi-tool workflows** | preferred over plain function calling for complex workflows involving multiple tools, real-time updates, or model/vendor independence. |
| 🔴 | **Extra moving parts** | you now run and maintain servers, not just call an API directly; more surface area than a single function call. |
| 🔴 | **Overkill for simple cases** | for one API and one app, function calling is sufficient for simple API interactions, and MCP's setup can be unnecessary overhead. |
| 🔴 | **Security surface** | every connected server is a new access point into your data/tools, so permissions and trust boundaries matter more. |


---
## 4. Code Example

> **Goal:** This is a minimal, illustrative sketch (not a full working server) showing the shape of an MCP tool definition and how a client-side call looks:

In [1]:
# --- A tiny "MCP server" exposing one tool ---
from mcp.server import Server
from mcp.types import Tool, TextContent

server = Server("weather-server")

@server.list_tools()
async def list_tools():
    return [
        Tool(
            name="get_weather",
            description="Get current weather for a city",
            inputSchema={
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        )
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "get_weather":
        city = arguments["city"]
        # In real life: call a weather API here
        return [TextContent(type="text", text=f"It's sunny in {city}.")]

# --- On the client side, the host app would then: ---
# 1. discover this tool via list_tools()
# 2. let the LLM decide to call get_weather(city="Brussels")
# 3. send that call to the server via call_tool()
# 4. receive "It's sunny in Brussels." and feed it back to the LLM

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **MCP is a universal connector, not a new AI model.**
- **Client-server is the core architecture.**
- **It solves the M×N integration problem.**
- **It's an open, evolving standard.**
- **It shines for complex, multi-tool agents; it's overkill for a single simple API call.**

</div>